# Feature Importance for Tree Models

**Course:** BUSI70575 — Systematic Trading Strategies with ML
**Sources:** Session 2 (cluster-level MDI + PFI — centrepiece), Sessions 4/5/6 (per-feature).

Two families are used in the course:

* **MDI — Mean Decrease Impurity** (`.feature_importances_`): *in-sample*, read straight from the fitted
  trees — how much each feature reduced impurity across all its splits.
* **PFI — Permutation Feature Importance** (a.k.a. MDA): *out-of-sample* — shuffle a feature and measure
  how much the score drops.

> **SFI** (Single Feature Importance — train a model on one feature at a time) is *not* implemented in the
> course materials, so it is noted but not covered here.

Run top-to-bottom with the **`stml`** kernel.

---

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, log_loss
from sklearn.base import clone

np.random.seed(42)

## 1. MDI — Mean Decrease Impurity

When a node splits on feature $f$, the **weighted impurity decrease** is
$$\Delta i = \frac{N_{\text{node}}}{N}\,i_{\text{node}} - \frac{N_L}{N}\,i_L - \frac{N_R}{N}\,i_R,$$
where $i$ is the node impurity (entropy/Gini) and $N_\bullet$ are the (weighted) sample counts. For one tree,
$$\text{MDI}(f) = \!\!\sum_{\text{nodes splitting on }f}\!\! \Delta i,$$
normalised so the tree's importances sum to 1. The forest's importance is the **average** over trees.

`sklearn` computes exactly this from `tree_.impurity` and `tree_.weighted_n_node_samples`. It is **free**
(no extra passes) but **in-sample** and **biased toward high-cardinality / many-split features**.

### Worked example (by hand, Gini, 2 splits)

Root (100 samples, $G=0.5$) splits on **A** into L(60, $G=4/9$) and R(40, $G=0.5$); node R then splits on
**B** into (25, $G=0.32$) and (15, $G=4/9$). Using *unnormalised* weighted decreases (the $1/N$ cancels in
the final normalisation):

$$\Delta_A = 100(0.5) - 60(\tfrac49) - 40(0.5) = 3.333,\qquad
\Delta_B = 40(0.5) - 25(0.32) - 15(\tfrac49) = 5.333.$$

Normalising: $\text{MDI}(A)=\tfrac{3.333}{8.667}=\mathbf{0.385}$, $\text{MDI}(B)=\mathbf{0.615}$. B matters
more *even though it splits deeper* — MDI rewards total impurity removed, not position.

In [ ]:
# Hand-worked arithmetic (Gini 2-split tree)
dec_A = 100 * 0.5 - 60 * (4 / 9) - 40 * 0.5
dec_B = 40 * 0.5 - 25 * 0.32 - 15 * (4 / 9)
tot = dec_A + dec_B
print(f"A: raw={dec_A:.4f}  MDI={dec_A/tot:.4f}")
print(f"B: raw={dec_B:.4f}  MDI={dec_B/tot:.4f}")
assert np.isclose(dec_A / tot, 0.3846, atol=1e-3)
assert np.isclose(dec_B / tot, 0.6154, atol=1e-3)
print("OK: matches hand-worked MDI")

In [ ]:
# Reproduce sklearn's .feature_importances_ from raw tree internals.
cols = ["sig0", "sig1", "sig2", "noise0", "noise1", "noise2"]
Xfi, yfi = make_classification(n_samples=1000, n_features=6, n_informative=3,
                               n_redundant=0, n_repeated=0, shuffle=False, random_state=0)
Xfi = pd.DataFrame(Xfi, columns=cols); yfi = pd.Series(yfi)
X_tr, X_te, y_tr, y_te = train_test_split(Xfi, yfi, test_size=0.3, random_state=0)

rf = RandomForestClassifier(n_estimators=100, max_depth=6, max_features="sqrt",
                            criterion="entropy", random_state=42, n_jobs=-1).fit(X_tr, y_tr)

def tree_mdi(tree):
    t = tree.tree_
    imp = np.zeros(t.n_features)
    for node in range(t.node_count):
        if t.children_left[node] != t.children_right[node]:          # internal (split) node
            f = t.feature[node]
            l, r = t.children_left[node], t.children_right[node]
            imp[f] += (t.weighted_n_node_samples[node] * t.impurity[node]
                       - t.weighted_n_node_samples[l] * t.impurity[l]
                       - t.weighted_n_node_samples[r] * t.impurity[r])
    imp /= t.weighted_n_node_samples[0]      # divide by total weighted samples (root)
    return imp / imp.sum()                    # normalise so the tree's importances sum to 1

assert np.allclose(tree_mdi(rf.estimators_[0]), rf.estimators_[0].feature_importances_)   # one tree
manual = np.mean([tree_mdi(est) for est in rf.estimators_], axis=0)                        # forest = mean
assert np.allclose(manual, rf.feature_importances_)
print("OK: reproduced .feature_importances_ from tree internals")

mdi = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
print(mdi.round(4).to_string())

## 2. PFI — Permutation Feature Importance (MDA)

Fit once. On held-out data, record the baseline score, then **shuffle one feature's column** (breaking its
link to the target while keeping its marginal distribution) and re-score. The importance is the average
drop over `n_repeats`:
$$\text{PFI}(f) = \text{score}(X) - \tfrac{1}{R}\sum_{r=1}^{R}\text{score}\!\left(X^{\text{perm}_r}_f\right).$$

It is **model-agnostic**, **out-of-sample**, and **not** cardinality-biased — but it is **expensive** and,
for correlated features, *under*-states importance (a shuffled feature is "covered" by its twins). A useless
feature scores $\approx 0$ and can go **negative** by chance.

Session 2's CV variant uses negative log-loss and the normalised drop
`importance = (loss_perm - loss_base) / loss_perm`.

In [ ]:
r = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=42, scoring="accuracy")
pfi = pd.DataFrame({"feature": cols, "importance": r.importances_mean,
                    "std": r.importances_std}).sort_values("importance", ascending=False)
print(pfi.round(4).to_string(index=False))

imp = pfi.set_index("feature")["importance"]
assert imp[["sig0", "sig1", "sig2"]].mean() > imp[["noise0", "noise1", "noise2"]].mean()
print("\nNote: noise features sit near 0 and can be slightly NEGATIVE (chance).")

In [ ]:
# Reproduce the permutation idea by hand and confirm the signal>noise ordering.
def manual_pfi(model, X, y, feature, n_repeats=20, seed=0):
    rng = np.random.default_rng(seed)
    base = accuracy_score(y, model.predict(X))
    drops = [base - accuracy_score(y, model.predict(
                 X.assign(**{feature: rng.permutation(X[feature].values)})))
             for _ in range(n_repeats)]
    return float(np.mean(drops))

top_sig = mdi.index[0]                                    # strongest feature (a signal)
m_sig = manual_pfi(rf, X_te, y_te, top_sig)
m_noise = manual_pfi(rf, X_te, y_te, "noise0")
print(f"manual PFI  {top_sig}={m_sig:.4f}   noise0={m_noise:.4f}")
assert m_sig > m_noise
print("OK: manual permutation reproduces the signal>noise ordering")

## 3. Cluster-level importance (Session 2)

Correlated features cause **substitution effects**: MDI splits one signal's importance across its copies,
and PFI hides it (shuffling one copy leaves the others intact). The fix is to score *groups*:

* **Cluster MDI** — sum the per-feature MDI within each cluster (per tree), then average across trees.
* **Cluster PFI** — permute *all* features in a cluster **with the same permutation**, preserving
  within-cluster correlation while breaking the cluster↔target link.

Below: 3 correlated copies of signal **A**, 2 of signal **B** (A drives the target more), and 3 pure-noise
features. Per-feature MDI dilutes A across A0/A1/A2; clustering recovers it.

In [ ]:
# --- Verbatim from Solution_Programming_Session_2.ipynb (cell 60) ---
def calculate_cluster_importance_mdi(model, feature_names, clusters):
    """
    Calculate Mean Decrease Impurity (MDI) feature importance at the cluster level.
    Returns a DataFrame with mean and std of cluster importance.
    """
    if hasattr(model, 'estimators_'):
        importances = {i: tree.feature_importances_
                      for i, tree in enumerate(model.estimators_)}
    else:
        importances = {0: model.feature_importances_}

    imp_df = pd.DataFrame.from_dict(importances, orient='index')
    imp_df.columns = feature_names
    imp_df = imp_df.replace(0, np.nan)            # zeros happen when max_features=1

    cluster_importance = pd.DataFrame(columns=['mean', 'std'])
    for cluster_id, features in clusters.items():
        valid_features = [f for f in features if f in feature_names]
        if valid_features:
            cluster_imp = imp_df[valid_features].sum(axis=1)
            cluster_importance.loc[f'Cluster_{cluster_id}', 'mean'] = cluster_imp.mean()
            if len(cluster_imp) > 1:
                cluster_importance.loc[f'Cluster_{cluster_id}', 'std'] = (
                    cluster_imp.std() * cluster_imp.shape[0]**-0.5)
            else:
                cluster_importance.loc[f'Cluster_{cluster_id}', 'std'] = 0

    total_importance = cluster_importance['mean'].sum()
    if total_importance > 0:
        cluster_importance['mean'] = cluster_importance['mean'] / total_importance
    return cluster_importance


# --- Verbatim from Solution_Programming_Session_2.ipynb (cell 62) ---
def calculate_cluster_importance_pfi(model, X, y, clusters, cv=5, scoring='neg_log_loss'):
    """
    Permutation feature importance (PFI) at the cluster level (CV + negative log-loss).
    Returns a DataFrame with mean and std of cluster importance.
    """
    from sklearn.model_selection import KFold
    from sklearn.metrics import log_loss

    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    baseline_scores = pd.Series(dtype='float64')
    permutation_scores = pd.DataFrame(columns=clusters.keys())

    for i, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_test)
        baseline_scores.loc[i] = -log_loss(y_test, y_pred, labels=model.classes_)

        for cluster_id in clusters.keys():
            X_test_permuted = X_test.copy()
            cluster_features = [f for f in clusters[cluster_id] if f in X.columns]
            if cluster_features:
                # same permutation for all features in the cluster -> preserves within-cluster corr
                permutation_idx = np.random.permutation(len(X_test))
                for feature in cluster_features:
                    X_test_permuted[feature] = X_test_permuted[feature].values[permutation_idx]
                y_pred_permuted = model.predict_proba(X_test_permuted)
                permutation_scores.loc[i, cluster_id] = -log_loss(
                    y_test, y_pred_permuted, labels=model.classes_)

    importance = (-1 * permutation_scores).add(baseline_scores, axis=0)
    importance = importance / (-1 * permutation_scores)
    cluster_importance = pd.DataFrame({
        'mean': importance.mean(),
        'std': importance.std() * importance.shape[0]**-0.5})
    cluster_importance.index = [f'Cluster_{i}' for i in cluster_importance.index]
    return cluster_importance

In [ ]:
def make_clustered(n=800, seed=0):
    rng = np.random.default_rng(seed)
    sigA, sigB = rng.normal(size=n), rng.normal(size=n)
    p = 1 / (1 + np.exp(-(1.3 * sigA + 0.7 * sigB)))     # A drives the target more than B
    y = (rng.uniform(size=n) < p).astype(int)
    data = {}
    for k in range(3): data[f"A{k}"] = sigA + 0.10 * rng.normal(size=n)   # 3 correlated copies of A
    for k in range(2): data[f"B{k}"] = sigB + 0.10 * rng.normal(size=n)   # 2 correlated copies of B
    for k in range(3): data[f"N{k}"] = rng.normal(size=n)                 # 3 pure-noise features
    return pd.DataFrame(data), pd.Series(y)

Xc2, yc2 = make_clustered()
clusters = {0: ["A0", "A1", "A2"], 1: ["B0", "B1"], 2: ["N0", "N1", "N2"]}

rf_c = RandomForestClassifier(n_estimators=200, max_depth=6, max_features="sqrt",
                              criterion="entropy", random_state=42, n_jobs=-1).fit(Xc2, yc2)

per_feat = pd.Series(rf_c.feature_importances_, index=Xc2.columns)
print("per-feature MDI (signal A is split across A0/A1/A2):")
print(per_feat.round(4).to_string(), "\n")

np.random.seed(42)                                   # the PFI function uses the global RNG
mdi_cl = calculate_cluster_importance_mdi(rf_c, list(Xc2.columns), clusters)
pfi_cl = calculate_cluster_importance_pfi(clone(rf_c), Xc2, yc2, clusters)
print("cluster MDI (sums to 1):")
print(mdi_cl.round(4).to_string(), "\n")
print("cluster PFI (normalised drop):")
print(pfi_cl.round(4).to_string())

assert mdi_cl.loc["Cluster_0", "mean"] > mdi_cl.loc["Cluster_2", "mean"]        # A-signal > noise
assert mdi_cl.loc["Cluster_0", "mean"] > per_feat[["A0", "A1", "A2"]].max()     # cluster recovers signal
print("\nOK: clustering recovers signal A that per-feature MDI diluted across A0/A1/A2")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
per_feat.plot.bar(ax=ax[0], color="steelblue", title="per-feature MDI (A diluted)")
ax[0].set_ylabel("importance")
mdi_cl["mean"].astype(float).plot.bar(ax=ax[1], color="indianred", title="cluster MDI (A recovered)")
ax[1].set_ylabel("importance")
plt.tight_layout(); plt.show()

## 4. MDI vs PFI — when they disagree

| aspect | MDI | PFI (permutation / MDA) |
|---|---|---|
| data used | in-sample (training) | out-of-sample (held-out / CV) |
| measures | total impurity decrease from splits | drop in score when a feature is shuffled |
| bias | inflated for high-cardinality / many-split features | ~unbiased, model-agnostic |
| correlated features | importance split among copies (dilution) | shuffling one leaves twins intact → both look weak |
| can be negative? | no ($\ge 0$) | yes (noise can "help" by chance) |
| cost | free (read from fitted trees) | expensive (re-score per feature × repeats) |
| fix for correlation | **cluster MDI** (sum within cluster) | **cluster PFI** (permute the whole cluster together) |

**Rule of thumb:** use MDI for a fast first look, PFI (out-of-sample) to confirm, and cluster-level versions
whenever features are correlated — exactly the Session 2 workflow.

## Source pointers

| What | File | Cell |
|---|---|---|
| per-feature MDI bar | `Solution_Programming_Session_2.ipynb` | 35 |
| per-feature PFI bar | `Solution_Programming_Session_2.ipynb` | 37 |
| `calculate_cluster_importance_mdi` | `Solution_Programming_Session_2.ipynb` | 60 |
| `calculate_cluster_importance_pfi` | `Solution_Programming_Session_2.ipynb` | 62 |
| MDI for RF / XGB | `Solution_Programming_Session_4/5.ipynb` | 47, 49 |
| `plot_permutation_importance` | `Solution_Programming_Session_5.ipynb` | 64 |

Reference: M. López de Prado, *Advances in Financial Machine Learning*, Ch. 8 (MDI, MDA, SFI).